In [0]:
%pip install imbalanced-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Bibliotecas utilizadas na Construção da Aplicação
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [0]:
# Carrega os dados da tabela Unity Catalog
data = spark.table('workspace.default.fraud').toPandas()

In [0]:
data

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,42,PAYMENT,5660.56,C618230452,0.00,0.00,M1352937915,0.00,0.00,0,0
1,42,CASH_IN,107551.46,C163660215,300191.00,407742.46,C470834836,223824.28,116272.81,0,0
2,42,TRANSFER,599918.25,C1862740146,158527.00,0.00,C161137214,0.00,599918.25,0,0
3,42,PAYMENT,5899.22,C10795828,80252.00,74352.78,M1903979378,0.00,0.00,0,0
4,42,CASH_OUT,10599.52,C1924354067,10908.00,308.48,C2136594180,0.00,10599.52,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [0]:
# Análise Exploratória dos Dados
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("📊 ANÁLISE EXPLORATÓRIA DE DADOS - FRAUDE")
print("="*60)

# Informações básicas
print("\n1️⃣ INFORMAÇÕES GERAIS:")
print(f"   Dimensões: {data.shape[0]} linhas x {data.shape[1]} colunas")
print(f"   Período: step {data['step'].min()} a {data['step'].max()}")

# Valores faltantes
print("\n2️⃣ VALORES FALTANTES:")
missing = data.isnull().sum()
if missing.sum() == 0:
    print("   ✓ Nenhum valor faltante!")
else:
    print(missing[missing > 0])

# Distribuição da variável target
print("\n3️⃣ DISTRIBUIÇÃO DE FRAUDES:")
fraud_counts = data['isFraud'].value_counts()
print(fraud_counts)
print(f"\n   Taxa de fraude: {(fraud_counts[1] / len(data) * 100):.2f}%")
print(f"   ⚠️ Desbalanceamento: {(fraud_counts[0] / fraud_counts[1]):.1f}:1")

# Estatísticas por tipo de transação
print("\n4️⃣ FRAUDES POR TIPO DE TRANSAÇÃO:")
fraud_by_type = data.groupby('type')['isFraud'].agg(['sum', 'count', 'mean'])
fraud_by_type.columns = ['Total Fraudes', 'Total Transações', 'Taxa Fraude']
fraud_by_type['Taxa Fraude'] = (fraud_by_type['Taxa Fraude'] * 100).round(2)
print(fraud_by_type.sort_values('Total Fraudes', ascending=False))

print("\n" + "="*60)

📊 ANÁLISE EXPLORATÓRIA DE DADOS - FRAUDE

1️⃣ INFORMAÇÕES GERAIS:
   Dimensões: 6362620 linhas x 11 colunas
   Período: step 1 a 743

2️⃣ VALORES FALTANTES:
   ✓ Nenhum valor faltante!

3️⃣ DISTRIBUIÇÃO DE FRAUDES:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

   Taxa de fraude: 0.13%
   ⚠️ Desbalanceamento: 773.7:1

4️⃣ FRAUDES POR TIPO DE TRANSAÇÃO:
          Total Fraudes  Total Transações  Taxa Fraude
type                                                  
CASH_OUT           4116           2237500         0.18
TRANSFER           4097            532909         0.77
CASH_IN               0           1399284         0.00
DEBIT                 0             41432         0.00
PAYMENT               0           2151495         0.00



In [0]:
# Explorando o Tipo de Transação "type"
print(data.type.value_counts())

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [0]:
# Explorando o Tipo de Transação "type"
type = data["type"].value_counts()

In [0]:
# Explorando o Tipo de Transação "type" e Criando Gráfico de Rosca
type = data["type"].value_counts()
transactions = type.index
quantity = type.values

import plotly.express as px
figure = px.pie(data, 
             values=quantity, 
             names=transactions,hole = 0.5, 
             title="Distribution of Transaction Type")
figure.show()

In [0]:
# Checking correlation - Checando as Correlações do Target
# Create a temporary numeric version of isFraud for correlation analysis
data_numeric = data.copy()
if data_numeric['isFraud'].dtype == 'object':
    data_numeric['isFraud'] = data_numeric['isFraud'].map({"No Fraud": 0, "Fraud": 1})

# Select only numeric columns for correlation
correlation = data_numeric.select_dtypes(include=[np.number]).corr()
print(correlation["isFraud"].sort_values(ascending=False))

isFraud           1.000000
amount            0.076688
isFlaggedFraud    0.044109
step              0.031578
oldbalanceOrg     0.010154
newbalanceDest    0.000535
oldbalanceDest   -0.005885
newbalanceOrig   -0.008148
type                   NaN
Name: isFraud, dtype: float64


In [0]:
# Fazendo conversão de object para número
data["type"] = data["type"].map({"CASH_OUT": 1, "PAYMENT": 2, 
                                 "CASH_IN": 3, "TRANSFER": 4,
                                 "DEBIT": 5})


In [0]:
# Alterando a Label (teste para a saída ficar visível)
data["isFraud"] = data["isFraud"].map({0: "No Fraud", 1: "Fraud"})
print(data.head())

   step  type     amount  ... newbalanceDest   isFraud  isFlaggedFraud
0    42   NaN    5660.56  ...           0.00  No Fraud               0
1    42   NaN  107551.46  ...      116272.81  No Fraud               0
2    42   NaN  599918.25  ...      599918.25  No Fraud               0
3    42   NaN    5899.22  ...           0.00  No Fraud               0
4    42   NaN   10599.52  ...       10599.52  No Fraud               0

[5 rows x 11 columns]


In [0]:
data

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,42,NaN,5660.56,C618230452,0.00,0.00,M1352937915,0.00,0.00,No Fraud,0
1,42,NaN,107551.46,C163660215,300191.00,407742.46,C470834836,223824.28,116272.81,No Fraud,0
2,42,NaN,599918.25,C1862740146,158527.00,0.00,C161137214,0.00,599918.25,No Fraud,0
3,42,NaN,5899.22,C10795828,80252.00,74352.78,M1903979378,0.00,0.00,No Fraud,0
4,42,NaN,10599.52,C1924354067,10908.00,308.48,C2136594180,0.00,10599.52,No Fraud,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,NaN,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,Fraud,0
6362616,743,NaN,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,Fraud,0
6362617,743,NaN,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,Fraud,0
6362618,743,NaN,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,Fraud,0


In [0]:
# Estatística dos Campos
data.describe().T

,count,mean,std,min,25%,50%,75%,max
step,6362620.0,2.433972e+02,1.423320e+02,1.0,156.00,239.000,3.350000e+02,7.430000e+02
type,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,13389.57,74871.940,2.087215e+05,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.00,14208.000,1.073152e+05,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.00,0.000,1.442584e+05,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.00,132705.665,9.430367e+05,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.00,214661.440,1.111909e+06,3.561793e+08
isFlaggedFraud,6362620.0,2.514687e-06,1.585775e-03,0.0,0.00,0.000,0.000000e+00,1.000000e+00


In [0]:
# Avaliando o Target
data.isFraud.value_counts()

isFraud
No Fraud    6354407
Fraud          8213
Name: count, dtype: int64

In [0]:
# Separando as Variáveis Explicativas (x) da variável Target (y)
from sklearn.model_selection import train_test_split
x = np.array(data[["type", "amount", "oldbalanceOrg", "newbalanceOrig"]])
y = np.array(data[["isFraud"]])

In [0]:
# Treinando a Máquina Preditiva com Machine Learning
from sklearn.tree import DecisionTreeClassifier
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.10, random_state=42)
model = DecisionTreeClassifier()
model.fit(xtrain, ytrain)
#print(model.score(xtest, ytest))

DecisionTreeClassifier()

In [0]:
# Fazendo novas Predições com dados de Teste
y_pred = model.predict(xtest)

In [0]:
y_pred

array(['No Fraud', 'No Fraud', 'No Fraud', ..., 'No Fraud', 'No Fraud',
       'No Fraud'], dtype=object)

In [0]:
# Evaluate model - Avaliando a Máquina Preditiva (Modelo)
print('Métricas do Classification Report: \n', classification_report(ytest, y_pred))
print('Acurácia: \n', accuracy_score(ytest, y_pred))
print('Confusion Matrix: \n', confusion_matrix(ytest, y_pred))

Métricas do Classification Report: 
               precision    recall  f1-score   support

       Fraud       0.92      0.98      0.95       816
    No Fraud       1.00      1.00      1.00    635446

    accuracy                           1.00    636262
   macro avg       0.96      0.99      0.97    636262
weighted avg       1.00      1.00      1.00    636262

Acurácia: 
 0.9998585488367999
Confusion Matrix: 
 [[   799     17]
 [    73 635373]]


In [0]:
print('Acurácia: \n', accuracy_score(ytest, y_pred))

Acurácia: 
 0.9998585488367999


In [0]:
# prediction
#features = [type, amount, oldbalanceOrg, newbalanceOrig]
features = np.array([[4, 9000.60, 9000.60, 0.0]])
print(model.predict(features))

['Fraud']


In [0]:
# prediction
#features = [type, amount, oldbalanceOrg, newbalanceOrig]
features = np.array([[2, 5000, 5000, 0.0]])
print(model.predict(features))

['No Fraud']


In [0]:
# prediction
#features = [type, amount, oldbalanceOrg, newbalanceOrig]
features = np.array([[1, 5000, 5000, 0.0]])
print(model.predict(features))

['No Fraud']
